In [ ]:
%cd ..

/Users/lallannkhann/Documents/AIC26/backend


In [2]:
import json
from datetime import datetime
from dataclasses import asdict, is_dataclass

import torch
from src.services.processing.keyframe_extraction.service import KeyframeExtractionService
from src.services.processing.keyframe_extraction.interface import VideoShots, Shot
from src.services.processing.keyframe_extraction.utils import get_fps
from src.services.embedding.service import EmbeddingService

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
config = {
    "model_name": "laion/CLIP-ViT-L-14-laion2B-s32B-b82K",
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

embedding_service = EmbeddingService()
embedding_service.registry_model(
    config=config,
    provider_name="hf_clip_multimodal",
    model_name="CLIP-ViT-L-14"
)

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 5455.18it/s]


In [4]:
with open("sample/processed/shots/sample_video.json", "r", encoding="utf-8") as f:
    shot_data = json.load(f)
    
shots = [
    Shot(
        shot_index=shot['shot_index'],
        start_frame=shot['start_frame'],
        end_frame=shot['end_frame']
    )
    for shot in shot_data["shots"]
]


video_path = shot_data["video_path"]
if video_path.startswith("backend/"):
    video_path = video_path.replace("backend/", "", 1)
    
video_shots = VideoShots(
    video_path=video_path,
    video_fps=get_fps(video_path),
    shots=shots
)

In [5]:
keyframe_extraction_service = KeyframeExtractionService(
    min_keyframes_per_shot=1,
    max_keyframes_per_shot=100
)

keyframe_result = keyframe_extraction_service.extract_keyframes_from_shots(
    video_shots=video_shots,
    embedding_model=embedding_service.get_model(model_name="CLIP-ViT-L-14"),
    keyframe_ratio=0.1
)

Distilling keyframes: 100%|██████████| 169/169 [00:00<00:00, 27197.08it/s]


In [6]:
def default_serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    if is_dataclass(obj):
        return asdict(obj)
    raise TypeError(f"Type {type(obj)} not serializable")

with open("sample/processed/keyframes/sample_video.json", "w", encoding="utf-8") as f:
    json.dump(asdict(keyframe_result), f, default=default_serializer, ensure_ascii=False, indent=4)